# TS-GNN: Temporal Sheaf Graph Neural Network
**Allele-conditioned GRN rewiring via temporal sheaf diffusion on A100**

## Setup
1. **Runtime > Change runtime type > A100 GPU**
2. Esegui tutte le celle in ordine
3. Il progetto viene caricato via zip upload

In [ ]:
# ============================================================
# CELLA 0 - Verifica GPU
# ============================================================
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({vram:.0f} GB VRAM)")
    if 'A100' in gpu_name:
        print('A100 confermata!')
    else:
        print(f'ATTENZIONE: hai {gpu_name}, non A100. Vai su Runtime > Change runtime type')
else:
    print('NESSUNA GPU! Vai su Runtime > Change runtime type > A100')

In [ ]:
# ============================================================
# CELLA 1 - Upload progetto
# ============================================================
# Zip la cartella ts-gnn sul tuo PC e caricala qui.
# Su Windows: tasto destro sulla cartella ts-gnn > Comprimi in file ZIP
#
# Se preferisci Google Drive, decommenta l'Opzione B sotto.

# --- OPZIONE A: Upload diretto (consigliato) ---
import os
from google.colab import files

if not os.path.exists('/content/ts-gnn/src/tsgnn'):
    print('Carica il file ts-gnn.zip (o tsgnn_latest.zip):')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q -o "{zip_name}" -d /content/
    # Gestisci caso in cui lo zip contiene una sottocartella
    if not os.path.exists('/content/ts-gnn/src') and os.path.exists('/content/ts-gnn/ts-gnn/src'):
        !mv /content/ts-gnn/ts-gnn/* /content/ts-gnn/
    # Gestisci caso in cui il nome della cartella nello zip e' diverso
    if not os.path.exists('/content/ts-gnn/src'):
        dirs = [d for d in os.listdir('/content') if os.path.isdir(f'/content/{d}') and os.path.exists(f'/content/{d}/src/tsgnn')]
        if dirs:
            !mv /content/{dirs[0]} /content/ts-gnn
    print(f'Progetto estratto in /content/ts-gnn')
else:
    print('Progetto gia presente in /content/ts-gnn')

# --- OPZIONE B: Google Drive (decommentare se serve) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/ts-gnn' /content/ts-gnn

PROJECT_DIR = '/content/ts-gnn'
!ls {PROJECT_DIR}/src/tsgnn/

In [ ]:
# ============================================================
# CELLA 2 - Installa dipendenze
# ============================================================
%%capture install_output
!pip install torch-geometric torch-sparse torch-scatter \
    scanpy scvelo anndata harmonypy scrublet \
    decoupler wandb hydra-core omegaconf \
    networkx plotly seaborn pyyaml tqdm \
    fair-esm GEOparse scipy

# Installa il progetto in modalita' editable
!cd {PROJECT_DIR} && pip install -e ".[all]"

print('Dipendenze installate!')

In [ ]:
# ============================================================
# CELLA 3 - Setup imports e paths
# ============================================================
import sys
import os
import time
import json
import logging
import random

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

# Assicurati che il progetto sia nel path
sys.path.insert(0, f'{PROJECT_DIR}/src')
os.chdir(PROJECT_DIR)

# Verifica imports
from tsgnn.model.tsgnn import TSGNN, create_tsgnn_from_config
from tsgnn.model.sheaf import SheafDiffusionLayer
from tsgnn.model.baselines import EvolveGCN, TemporalGAT, create_tsgnn_no_allele
from tsgnn.training.loss import TSGNNLoss
from tsgnn.training.trainer import TSGNNTrainer, create_synthetic_training_data
from tsgnn.evaluation.metrics import (
    evaluate_all, rewiring_distinguishability,
    extract_differential_edges, compute_hub_influence_scores
)
from tsgnn.evaluation.benchmarks import BenchmarkRunner
from tsgnn.visualization.attention import analyze_regulatory_mode
from tsgnn.visualization.rewiring import (
    plot_rewiring_trajectory, plot_differential_rewiring,
    identify_bottleneck_timepoints
)
from tsgnn.visualization.networks import plot_allele_embedding_space

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Tutti i moduli importati. Device: {DEVICE}')

In [ ]:
# ============================================================
# CELLA 4 - Configurazione
# ============================================================
with open(f'{PROJECT_DIR}/configs/default.yaml') as f:
    config = yaml.safe_load(f)

# Override per A100
config['training']['mixed_precision'] = True   # bf16 su A100
config['training']['max_epochs'] = 500
config['training']['patience'] = 20
config['logging']['use_wandb'] = False

# Parametri chiave
N = config['data']['n_genes']       # 500
d = config['model']['stalk_dim']    # 4
K = config['temporal']['K']         # 10
ESM_DIM = config['model']['esm_dim'] # 1280

# Riproducibilita'
SEED = config['seed']  # 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

print(f'Config: N={N}, d={d}, K={K}, ESM={ESM_DIM}')
print(f'Training: lr={config["training"]["lr"]}, epochs={config["training"]["max_epochs"]}, patience={config["training"]["patience"]}')
print(f'Loss: lambda_1={config["loss"]["lambda_1"]}, lambda_2={config["loss"]["lambda_2"]}, lambda_3={config["loss"]["lambda_3"]}')

## ESM-2 Allele Embeddings

In [ ]:
# ============================================================
# CELLA 5 - Carica o genera ESM-2 embeddings
# ============================================================
ALLELES = ['WT', 'R175H', 'R273H', 'R248W', 'R282W', 'G245S', 'Y220C']
TRAIN_ALLELES = ['WT', 'R175H', 'R273H', 'R248W']

esm_dir = f'{PROJECT_DIR}/data/external/esm2_embeddings'
esm_embeddings = {}

# Prova a caricare embeddings pre-calcolati
loaded = 0
if os.path.exists(esm_dir):
    for allele in ALLELES:
        path = f'{esm_dir}/{allele}.pt'
        if os.path.exists(path):
            esm_embeddings[allele] = torch.load(path, map_location='cpu', weights_only=True)
            loaded += 1

if loaded == len(ALLELES):
    print(f'Embeddings ESM-2 caricati da {esm_dir} ({loaded} alleli)')
    for a, e in esm_embeddings.items():
        print(f'  {a}: shape={e.shape}, norm={e.norm():.2f}')
else:
    print(f'Solo {loaded}/{len(ALLELES)} embeddings trovati. Genero con ESM-2...')
    try:
        import esm
        from tsgnn.data.allele import TP53_CANONICAL_SEQUENCE, create_mutant_sequence

        esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
        esm_model = esm_model.to(DEVICE).eval()
        batch_converter = alphabet.get_batch_converter()

        sequences = [('WT', TP53_CANONICAL_SEQUENCE)]
        for allele in ALLELES[1:]:
            sequences.append((allele, create_mutant_sequence(allele)))

        esm_embeddings = {}
        with torch.no_grad():
            for name, seq in sequences:
                _, _, tokens = batch_converter([(name, seq)])
                tokens = tokens.to(DEVICE)
                results = esm_model(tokens, repr_layers=[33])
                emb = results['representations'][33][0, 1:-1].mean(dim=0).cpu()
                esm_embeddings[name] = emb
                print(f'  {name}: shape={emb.shape}, norm={emb.norm():.2f}')

        # Salva per riuso
        os.makedirs(esm_dir, exist_ok=True)
        for name, emb in esm_embeddings.items():
            torch.save(emb, f'{esm_dir}/{name}.pt')
            diff = emb - esm_embeddings['WT']
            torch.save(diff, f'{esm_dir}/{name}_diff.pt')
        print('Embeddings salvati!')

        del esm_model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f'ESM-2 fallito: {e}')
        print('Uso embeddings random come fallback')
        torch.manual_seed(SEED)
        esm_embeddings = {a: torch.randn(ESM_DIM) for a in ALLELES}

train_esm = {a: esm_embeddings[a] for a in TRAIN_ALLELES}

## Preparazione dati e training

In [ ]:
# ============================================================
# CELLA 6 - Prepara dati (sintetici per ora)
# ============================================================
# TODO: Sostituire con dati reali da GSE178341
# Vedi sezione in fondo per il pipeline con dati reali.

E = 2000  # edges nel GRN
torch.manual_seed(SEED)
edge_index = torch.randint(0, N, (2, E))

train_data, val_data, _ = create_synthetic_training_data(
    N=N, E=E, K=K, input_dim=N, alleles=TRAIN_ALLELES,
)

print(f'Dati sintetici pronti:')
print(f'  Train: {len(train_data)} alleli x K={K} bins x N={N} geni')
print(f'  Val:   {len(val_data)} alleli')
print(f'  GRN:   {E} edges')
print(f'  Nota: sostituire con dati reali per risultati pubblicabili!')

In [ ]:
# ============================================================
# CELLA 7 - Crea modello TS-GNN
# ============================================================
model = TSGNN(
    num_nodes=N,
    num_edges=E,
    stalk_dim=d,
    input_dim=N,
    esm_dim=ESM_DIM,
    conditioning_dim=config['model']['conditioning_dim'],
    edge_index=edge_index,
    num_diffusion_steps=config['model']['num_diffusion_steps'],
    use_allele_conditioning=True,
    use_vectorized=True,  # versione vettorizzata per A100
)

params = model.count_parameters()
print(f'Parametri totali: {params["total"]:,}')
for name, count in params.items():
    if name != 'total':
        print(f'  {name}: {count:,}')

mem_mb = params['total'] * 4 / 1e6
lap_mb = (N * d) ** 2 * 4 / 1e6
print(f'\nMemoria modello: ~{mem_mb:.1f} MB')
print(f'Sheaf Laplacian: {N*d}x{N*d} = {lap_mb:.1f} MB per step')
print(f'Totale per epoca (K={K}, 4 alleli): ~{lap_mb * K * 4:.0f} MB peak')

In [ ]:
# ============================================================
# CELLA 8 - TRAINING
# ============================================================
trainer = TSGNNTrainer(
    model=model,
    config=config,
    device=DEVICE,
    checkpoint_dir=f'{PROJECT_DIR}/checkpoints',
    use_wandb=False,
)

print(f'Training su {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'Max epochs: {config["training"]["max_epochs"]}, Early stopping patience: {config["training"]["patience"]}')
print('=' * 60)

t0 = time.time()
results = trainer.train(
    train_data=train_data,
    val_data=val_data,
    esm_embeddings=train_esm,
)
elapsed = time.time() - t0

print(f'\n{"=" * 60}')
print(f'TRAINING COMPLETATO')
print(f'  Epoche: {results["epochs_trained"]}')
print(f'  Best val loss: {results["best_val_loss"]:.6f}')
print(f'  Tempo: {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  Tempo/epoca: {elapsed/results["epochs_trained"]:.2f}s')

In [ ]:
# ============================================================
# CELLA 9 - Learning curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(len(results['history']['train_loss']))
ax1.plot(epochs, results['history']['train_loss'], label='Train', lw=2)
ax1.plot(epochs, results['history']['val_loss'], label='Validation', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Learning Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

ax2.plot(epochs, results['history']['lr'], color='green', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('LR Schedule')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/figures/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluation

In [ ]:
# ============================================================
# CELLA 10 - Evaluation metriche
# ============================================================
eval_results = evaluate_all(
    model=model,
    test_data=val_data,
    esm_embeddings=train_esm,
    stalk_dim=d,
)

print('Risultati evaluation:')
print('=' * 60)
for metric, values in eval_results.items():
    print(f'\n{metric}:')
    if isinstance(values, dict):
        for k, v in values.items():
            if isinstance(v, float):
                print(f'  {k}: {v:.4f}')
            else:
                print(f'  {k}: {v}')
    else:
        print(f'  {values}')

In [ ]:
# ============================================================
# CELLA 11 - Analisi restriction maps e bottleneck
# ============================================================
model.eval()
allele_outputs = {}

with torch.no_grad():
    for allele in TRAIN_ALLELES:
        node_seq = val_data[allele]['node_features_seq'].to(DEVICE)
        allele_emb = train_esm[allele].to(DEVICE)
        preds, maps_traj, laps = model(node_seq, allele_emb)
        allele_outputs[allele] = {
            'predictions': [p.cpu() for p in preds],
            'maps_trajectory': [m.cpu() for m in maps_traj],
            'laplacians': [l.cpu() for l in laps],
        }

# Analisi modalita regolatorie
print('Modalita regolatorie per allele:')
for allele in TRAIN_ALLELES:
    maps = allele_outputs[allele]['maps_trajectory']
    modes = analyze_regulatory_mode(maps, edge_index)
    mode_counts = {}
    for v in modes.values():
        mode_counts[v['mode']] = mode_counts.get(v['mode'], 0) + 1
    print(f'  {allele}: {mode_counts}')

# Bottleneck analysis
print('\nBottleneck transitions (finestre terapeutiche):')
for allele in ['R175H', 'R273H']:
    laps = allele_outputs[allele]['laplacians']
    bottlenecks = identify_bottleneck_timepoints(laps, edge_index, d)
    print(f'  {allele}:')
    for bn in bottlenecks[:3]:
        print(f'    t={bn["time_index"]}: centrality_change={bn["centrality_change"]:.4f}')

## Visualizzazioni

In [ ]:
# ============================================================
# CELLA 12 - Visualizzazioni
# ============================================================
FIG_DIR = f'{PROJECT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Rewiring trajectory
for allele in ['R175H', 'R273H']:
    try:
        plot_rewiring_trajectory(
            allele_outputs[allele]['laplacians'],
            edge_index, d, allele=allele, output_dir=FIG_DIR,
        )
    except Exception as e:
        print(f'  Rewiring plot {allele} fallito: {e}')

# Differential rewiring R175H vs R273H
try:
    plot_differential_rewiring(
        allele_outputs['R175H']['laplacians'],
        allele_outputs['R273H']['laplacians'],
        edge_index, d,
        allele_a='R175H', allele_b='R273H', output_dir=FIG_DIR,
    )
except Exception as e:
    print(f'  Differential plot fallito: {e}')

# Allele embedding space
try:
    plot_allele_embedding_space(esm_embeddings, method='tsne', output_dir=FIG_DIR)
except Exception as e:
    print(f'  Embedding plot fallito: {e}')

# Mostra tutti i plot
import glob
from IPython.display import display, Image as IPImage
for png in sorted(glob.glob(f'{FIG_DIR}/*.png')):
    print(f'\n{os.path.basename(png)}')
    display(IPImage(filename=png, width=700))

## Baseline Comparison

In [ ]:
# ============================================================
# CELLA 13 - Baselines: crea e addestra
# ============================================================
baselines = {
    'TS-GNN (full)': model,
}

# EvolveGCN
print('Training EvolveGCN...')
evolve_gcn = EvolveGCN(
    num_nodes=N, input_dim=N, hidden_dim=d*4, edge_index=edge_index,
)
evolve_trainer = TSGNNTrainer(
    model=evolve_gcn, config={**config, 'training': {**config['training'], 'max_epochs': 100, 'patience': 10}},
    device=DEVICE, use_wandb=False,
)
evolve_trainer.train(train_data, val_data, train_esm)
baselines['EvolveGCN'] = evolve_gcn

# TS-GNN no allele (ablation)
print('\nTraining TS-GNN (no allele)...')
no_allele = create_tsgnn_no_allele(
    num_nodes=N, num_edges=E, stalk_dim=d, input_dim=N, edge_index=edge_index,
)
no_allele_trainer = TSGNNTrainer(
    model=no_allele, config={**config, 'training': {**config['training'], 'max_epochs': 100, 'patience': 10}},
    device=DEVICE, use_wandb=False,
)
no_allele_trainer.train(train_data, val_data, train_esm)
baselines['TS-GNN (no allele)'] = no_allele

# Temporal GAT
print('\nTraining Temporal GAT...')
temp_gat = TemporalGAT(
    num_nodes=N, input_dim=N, hidden_dim=d*4, edge_index=edge_index,
)
gat_trainer = TSGNNTrainer(
    model=temp_gat, config={**config, 'training': {**config['training'], 'max_epochs': 100, 'patience': 10}},
    device=DEVICE, use_wandb=False,
)
gat_trainer.train(train_data, val_data, train_esm)
baselines['Temporal GAT'] = temp_gat

print(f'\nModelli addestrati: {list(baselines.keys())}')
for name, m in baselines.items():
    n_params = sum(p.numel() for p in m.parameters())
    print(f'  {name}: {n_params:,} parametri')

In [ ]:
# ============================================================
# CELLA 14 - Benchmark comparison
# ============================================================
print('Running benchmark comparison...')
runner = BenchmarkRunner(
    models=baselines,
    test_data=val_data,
    esm_embeddings=train_esm,
    stalk_dim=d,
    seeds=[42],
)

bench_results = runner.run_all()

print('\n' + '=' * 70)
print('BENCHMARK RESULTS')
print('=' * 70)
for model_name, metrics in bench_results.items():
    print(f'\n{model_name}:')
    for metric, val in metrics.items():
        if isinstance(val, dict) and 'mean' in val:
            print(f'  {metric}: {val["mean"]:.4f} +/- {val.get("std", 0):.4f}')
        elif isinstance(val, (int, float)):
            print(f'  {metric}: {val:.4f}')
        else:
            print(f'  {metric}: {val}')

## Ablation Studies

In [ ]:
# ============================================================
# CELLA 15 - Ablation: stalk dimension
# ============================================================
print('Ablation: stalk dimension d in {2, 4, 8}')
ablation_results = {}

for stalk_d in config['ablation']['stalk_dims']:
    print(f'\n--- d={stalk_d} ---')
    abl_model = TSGNN(
        num_nodes=N, num_edges=E, stalk_dim=stalk_d, input_dim=N,
        esm_dim=ESM_DIM, conditioning_dim=config['model']['conditioning_dim'],
        edge_index=edge_index, num_diffusion_steps=config['model']['num_diffusion_steps'],
    )
    abl_trainer = TSGNNTrainer(
        model=abl_model,
        config={**config, 'training': {**config['training'], 'max_epochs': 100, 'patience': 10}},
        device=DEVICE, use_wandb=False,
    )
    res = abl_trainer.train(train_data, val_data, train_esm)
    n_params = sum(p.numel() for p in abl_model.parameters())
    ablation_results[f'd={stalk_d}'] = {
        'val_loss': res['best_val_loss'],
        'epochs': res['epochs_trained'],
        'params': n_params,
    }
    print(f'  val_loss={res["best_val_loss"]:.6f}, params={n_params:,}')

print('\n--- Ablation Summary ---')
for k, v in ablation_results.items():
    print(f'{k}: val_loss={v["val_loss"]:.6f}, params={v["params"]:,}, epochs={v["epochs"]}')

## Salva risultati

In [ ]:
# ============================================================
# CELLA 16 - Salva tutto
# ============================================================
results_dir = f'{PROJECT_DIR}/results'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)

# Training history
with open(f'{results_dir}/training_history.json', 'w') as f:
    json.dump(results['history'], f, indent=2)

# Evaluation
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, torch.Tensor):
        return obj.tolist()
    return obj

with open(f'{results_dir}/evaluation.json', 'w') as f:
    json.dump(make_serializable(eval_results), f, indent=2)

with open(f'{results_dir}/benchmark.json', 'w') as f:
    json.dump(make_serializable(bench_results), f, indent=2)

with open(f'{results_dir}/ablation.json', 'w') as f:
    json.dump(make_serializable(ablation_results), f, indent=2)

# Checkpoint finale
torch.save(model.state_dict(), f'{PROJECT_DIR}/checkpoints/final_a100.pt')

print(f'Risultati salvati in {results_dir}/')
print(f'Checkpoint salvato in {PROJECT_DIR}/checkpoints/final_a100.pt')

# Download zip dei risultati
!cd {PROJECT_DIR} && zip -r /content/tsgnn_results.zip results/ figures/ checkpoints/final_a100.pt
from google.colab import files
files.download('/content/tsgnn_results.zip')
print('\nDownload avviato!')

In [ ]:
# ============================================================
# CELLA 17 - (Opzionale) Salva su Google Drive
# ============================================================
# Decommenta per salvare su Drive

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DIR = '/content/drive/MyDrive/ts-gnn-results'
# !mkdir -p {DRIVE_DIR}
# !cp -r {PROJECT_DIR}/results/* {DRIVE_DIR}/
# !cp -r {PROJECT_DIR}/figures/* {DRIVE_DIR}/
# !cp {PROJECT_DIR}/checkpoints/final_a100.pt {DRIVE_DIR}/
# print(f'Salvato su Drive: {DRIVE_DIR}')

In [ ]:
# ============================================================
# CELLA 18 - Unit tests (verifica integrita)
# ============================================================
!cd {PROJECT_DIR} && pip install -q pytest && python -m pytest tests/ -v --tb=short 2>&1 | tail -40

---
## Note per dati reali

Il training sopra usa **dati sintetici**. Per risultati pubblicabili:

1. **Scarica GSE178341** (CRC scRNA-seq) da GEO:
   ```python
   from tsgnn.data.download import download_crc_scrna
   download_crc_scrna()
   ```

2. **Preprocessing**:
   ```python
   from tsgnn.data.preprocess import preprocess_scrna, select_features
   adata = preprocess_scrna('data/raw/GSE178341')
   ```

3. **GRN construction**:
   ```python
   from tsgnn.data.grn_construction import construct_base_grn
   edge_index, edge_weight = construct_base_grn(adata, gene_list)
   ```

4. **Temporal graphs**:
   ```python
   from tsgnn.data.temporal import construct_temporal_graphs
   temporal_data = construct_temporal_graphs(adata, edge_index, K=10)
   ```

5. Sostituire `create_synthetic_training_data()` nella Cella 6 con i dati reali.

### Hardware
- **T4 (free)**: OK per N=500, d=4. ~30-60 min training
- **A100 (Pro)**: Consigliato. ~10-20 min training
- Il preprocessing (pySCENIC) richiede ~25GB RAM -> Colab Pro